# 01 Preprocessing

Turns the sampled monthly-row subset into the cleaned dataset that the rest of the project consumes
(`train_1m_rows_clean.parquet`). Two kinds of issues are handled by **one** preprocessing pipeline:

- **Organic** issues the real data actually has: `-1` codes (validated as a real category, kept),
  inconsistent float-coded categorical formatting, a near-constant column, sparse columns.
- **Simulated** ingestion errors we deliberately inject **into the raw data before preprocessing**,
  so the same pipeline must handle injected and organic problems together: duplicate rows,
  mixed/invalid date formats, categorical case/whitespace noise, numbers stored as text, extra
  missingness, and outlier spikes.

We then compare the cleaned simulated output against the cleaned
non-simulated output to show what the pipeline recovered and what residual (lossy) differences
remain.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

LIGHT_MODE = True
N_SAMPLE_ROWS = 100_000

INTERIM = Path('../data/interim')
ROWS_PATH = INTERIM / 'train_1m_rows.parquet'
LABELS_PATH = INTERIM / 'train_1m_labels.parquet'
assert ROWS_PATH.exists(), 'Run scripts/preprocess_train_subset.py first.'

def read_rows_for_notebook(path, n_rows):
    if not LIGHT_MODE:
        return pd.read_parquet(path)
    return next(pq.ParquetFile(path).iter_batches(batch_size=n_rows)).to_pandas()

rows = read_rows_for_notebook(ROWS_PATH, N_SAMPLE_ROWS)
labels = pd.read_parquet(LABELS_PATH)
print(f'sample rows {rows.shape} | full rows {pq.ParquetFile(ROWS_PATH).metadata.num_rows:,} | target rate {labels.target.mean():.3f}')

sample rows (100000, 190) | full rows 1,000,012 | target rate 0.270


## Organic Issue: is `-1` missing, or a real category?

The data is anonymized with no dictionary, so we test rather than assume. We compare the default
rate of the `-1` group against the real categories and against the genuinely-missing (`NaN`) rows.
`-1` sits among the real categories (well below the `NaN` rate), so it is **kept** as a level.

In [2]:
labeled = rows.merge(labels[['customer_ID', 'target']], on='customer_ID', how='left')
sentinel_check = []
for col in ['D_64', 'D_117', 'D_126']:
    grp = labeled[col].fillna('<NaN>')
    rate = labeled.groupby(grp)['target'].mean()
    neg = '-1' if '-1' in rate.index else '-1.0'
    reals = rate.drop([neg, '<NaN>'], errors='ignore')
    sentinel_check.append({'column': col, 'neg1_rate': round(float(rate[neg]), 3),
                           'real_cat_min': round(float(reals.min()), 3),
                           'real_cat_max': round(float(reals.max()), 3),
                           'nan_rate': round(float(rate.get('<NaN>', float('nan'))), 3)})
print(f'overall target rate: {labeled.target.mean():.3f}')
print('-1 sits among the real categories and is well below the NaN rate -> keep it, do not impute.')
pd.DataFrame(sentinel_check)

overall target rate: 0.262
-1 sits among the real categories and is well below the NaN rate -> keep it, do not impute.


,column,neg1_rate,real_cat_min,real_cat_max,nan_rate
0,D_64,0.296,0.177,0.364,0.417
1,D_117,0.303,0.139,0.461,0.455
2,D_126,0.223,0.244,0.331,0.487


## The Pipeline: inject simulated errors, then preprocess

`inject()` corrupts a copy of the raw rows with common ingestion errors. `preprocess()` is the
single cleaning pipeline; it repairs both the injected errors and the organic ones:

- duplicate rows -> dropped
- mixed/invalid dates -> parsed (`errors='coerce'`; true garbage becomes `NaT`)
- numbers stored as text (`"$0.93"`) -> stripped and coerced to numeric
- categorical case/whitespace -> normalized; float-coded categoricals (`'1.0'`) -> `'1'`
- outlier spikes -> winsorized to the [1, 99] percentile
- near-constant `D_87` -> dropped

Injected **missingness is left as `NaN`** on purpose: once injected we cannot tell it apart from the
organic missingness, and imputing everything here would destroy the predictive missingness signal
(EDA) and leak. Imputation stays in the modeling pipeline, fit per fold.

In [3]:
CATEGORICAL_COLS = ['D_63', 'D_64', 'D_66', 'D_68', 'B_30', 'B_31', 'B_38',
                    'D_114', 'D_116', 'D_117', 'D_120', 'D_126']
FLOAT_CODED_CATS = ['D_66', 'D_68', 'B_30', 'B_38', 'D_114', 'D_116', 'D_117', 'D_120', 'D_126']
DROP_COLS = ['D_87']

def numeric_columns(df):
    return [c for c in df.columns if c not in (['customer_ID', 'S_2'] + CATEGORICAL_COLS + DROP_COLS)]

def inject(df, rng):
    """Simulate messy ingestion on a COPY of the raw rows (before any preprocessing)."""
    df = df.copy()
    counts = {}
    m = len(df)
    di = rng.choice(m, int(0.01 * m), replace=False); h = len(di) // 2
    s2 = df['S_2'].dt.strftime('%Y-%m-%d').astype('object')
    s2.iloc[di[:h]] = df['S_2'].dt.strftime('%m/%d/%Y').to_numpy()[di[:h]]   # alternate format
    s2.iloc[di[h:]] = 'not a date'                                          # unparseable
    df['S_2'] = s2; counts['mixed/invalid dates'] = len(di)
    ci = rng.choice(m, int(0.01 * m), replace=False)
    c = df['D_63'].astype('object')
    c.iloc[ci] = [f' {str(v).lower()} ' if pd.notna(v) else v for v in c.iloc[ci]]
    df['D_63'] = c; counts['categorical case/space'] = len(ci)
    pi = rng.choice(m, int(0.005 * m), replace=False)
    p = df['P_2'].astype('object')
    p.iloc[pi] = [f'${v:.3f}' if pd.notna(v) else v for v in p.iloc[pi]]
    df['P_2'] = p; counts['numbers stored as text'] = len(pi)
    mi = rng.choice(m, int(0.01 * m), replace=False)
    df.iloc[mi, df.columns.get_loc('B_2')] = np.nan; counts['injected missingness'] = len(mi)
    oi = rng.choice(m, int(0.005 * m), replace=False)                      # spikes occupy top 0.5%; the 1/99 winsor cut sits below them
    df.iloc[oi, df.columns.get_loc('B_1')] = float(df['B_1'].max()) * 1000
    counts['outlier spikes'] = len(oi)
    # duplicate LAST so the copies are exact -> drop_duplicates removes them fully
    k = int(0.005 * m)
    df = pd.concat([df, df.sample(k, random_state=int(rng.integers(1e9)))], ignore_index=True)
    counts['duplicate rows'] = k
    return df, counts

def preprocess(df):
    """Single cleaning pipeline: handles injected AND organic issues."""
    df = df.drop_duplicates().copy()
    if df['S_2'].dtype == object:                                           # only if dates were corrupted
        df['S_2'] = pd.to_datetime(df['S_2'], errors='coerce', format='mixed')
    num = numeric_columns(df)
    for col in num:
        if df[col].dtype == object:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace(r'[^0-9eE.+-]', '', regex=True),
                                    errors='coerce')
    for col in CATEGORICAL_COLS:
        if col in df.columns:
            df[col] = df[col].astype('string').str.strip().str.upper()
    for col in FLOAT_CODED_CATS:
        if col in df.columns:
            df[col] = df[col].str.replace(r'\.0$', '', regex=True)
    for col in num:
        lo, hi = df[col].quantile([0.01, 0.99])
        df[col] = df[col].clip(lo, hi).astype('float32')
    return df.drop(columns=[c for c in DROP_COLS if c in df.columns])

print('inject() and preprocess() defined')

inject() and preprocess() defined


## Build the cleaned dataset

We build two cleaned outputs from the full raw rows: **non-simulated** (organic issues only) and
**simulated** (raw -> inject -> preprocess). The **simulated** output is saved as
`train_1m_rows_clean.parquet` — the dataset every downstream notebook consumes — so the preprocessing
is genuinely load-bearing.

In [4]:
CLEAN_PATH = INTERIM / 'train_1m_rows_clean.parquet'

def profile(df):
    return {'rows': int(len(df)), 'columns': int(df.shape[1]),
            'total_NaN_cells': int(df.isna().to_numpy().sum()),
            'overall_NaN_rate': round(float(df.isna().to_numpy().mean()), 5),
            'S_2_NaT': int(df['S_2'].isna().sum()),
            'B_2_NaN (injected-miss col)': int(df['B_2'].isna().sum()),
            'B_1_max (outlier col)': round(float(df['B_1'].max()), 3),
            'D_63_levels (case col)': int(df['D_63'].nunique())}

full = pd.read_parquet(ROWS_PATH)
rng = np.random.default_rng(5241)

clean_nonsim = preprocess(full)
prof_nonsim = profile(clean_nonsim)
del clean_nonsim

corrupted, inj_counts = inject(full, rng)
del full
clean_sim = preprocess(corrupted)
del corrupted
prof_sim = profile(clean_sim)

for col in CATEGORICAL_COLS:
    if col in clean_sim.columns:
        clean_sim[col] = clean_sim[col].astype('string')
clean_sim.to_parquet(CLEAN_PATH, compression='zstd')

sim_comparison = pd.DataFrame({'non_simulated': prof_nonsim, 'simulated': prof_sim})
corrections = {'rows_out': prof_sim['rows'], 'columns': prof_sim['columns'],
               'minus1_kept_as_category': ['D_64', 'D_117', 'D_126'],
               'categorical_formatting_normalized': FLOAT_CODED_CATS,
               'dropped_near_constant': DROP_COLS,
               'simulated_corruptions_injected': inj_counts,
               'output': str(CLEAN_PATH)}
(INTERIM / 'train_1m_corrections.json').write_text(json.dumps(corrections, indent=2) + '\n')
del clean_sim
print('injected:', inj_counts)
print(f'wrote {CLEAN_PATH} ({prof_sim["rows"]:,} rows, {prof_sim["columns"]} cols)')

injected: {'mixed/invalid dates': 10000, 'categorical case/space': 10000, 'numbers stored as text': 5000, 'injected missingness': 10000, 'outlier spikes': 5000, 'duplicate rows': 5000}
wrote ../data/interim/train_1m_rows_clean.parquet (1,000,012 rows, 189 cols)


## Simulated vs Non-Simulated Comparison

The pipeline **fully recovers** the reversible corruptions: identical row count (duplicates dropped) and identical `D_63` level count (case/whitespace normalized). Outlier spikes are winsorized away (the `B_1` cap returns to the organic scale, ~1 vs the injected ~1300). The residual differences are the **lossy** injections: extra `NaT` dates (true garbage), extra `B_2` `NaN`s (injected missingness, left for the leak-free modeling imputer), and a slightly higher `B_1` cap because injecting spikes overwrote some genuine values — the expected, honest footprint.

In [5]:
sim_comparison.to_csv(INTERIM / 'train_1m_sim_comparison.csv')
sim_comparison

,non_simulated,simulated
rows,1.000012e+06,1.000012e+06
columns,1.890000e+02,1.890000e+02
total_NaN_cells,2.809740e+07,2.811239e+07
overall_NaN_rate,1.486600e-01,1.487400e-01
S_2_NaT,0.000000e+00,5.000000e+03
B_2_NaN (injected-miss col),3.730000e+02,1.036500e+04
B_1_max (outlier col),1.011000e+00,1.140000e+00
D_63_levels (case col),6.000000e+00,6.000000e+00
